In [1]:
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import os
import random

In [2]:
ESC50_AUDIO_PATH = "../data/ESC-50"
ESC50_METADATA_PATH = "../data/ESC-50/esc50.csv"

BIO_CLASSES = [
    'dog', 'chirping_birds', 'crow', 'frog', 'cow', 'hen', 'insects', 'sheep', 
    'pig', 'rooster', 'cat', 'crickets', 'crying_baby', 'breathing', 
    'coughing', 'sneezing', 'snoring', 'laughing', 'clapping'
]

GEO_CLASSES = [
    'sea_waves', 'rain', 'wind', 'thunderstorm', 'water_drops', 'crackling_fire'
]

ANTRO_CLASSES = [
    'helicopter', 'chainsaw', 'siren', 'car_horn', 'engine', 'train', 'church_bells', 
    'airplane', 'fireworks', 'hand_saw', 'keyboard_typing', 'mouse_click', 
    'footsteps', 'door_wood_knock', 'door_wood_creaks', 'clock_alarm', 
    'clock_tick', 'glass_breaking', 'brushing_teeth', 'toilet_flush', 
    'washing_machine', 'vacuum_cleaner', 'can_opening', 'drinking_sipping',
    'pouring_water'
]

In [3]:
SAMPLE_RATE = 32000
DURATION_SEC = 5 
DURATION_SAMPLES = SAMPLE_RATE * DURATION_SEC

In [4]:
def load_audio(file_path, target_samples):

    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE, mono=True)
    
    if len(signal) < target_samples:
        # preenche com silêncio
        padding = target_samples - len(signal)
        signal = np.pad(signal, (0, padding), 'constant')
    elif len(signal) > target_samples:
        # trunca
        signal = signal[:target_samples]
        
    return signal


In [5]:
def normalize_rms(signal):
    epsilon = 1e-10
    rms = np.sqrt(np.mean(signal**2))
    target_rms = 0.1 
    return signal * (target_rms / (rms + epsilon))

In [6]:
def build_file_lists():
    dicio = {}

    df_esc50 = pd.read_csv(ESC50_METADATA_PATH)
    for index, row in df_esc50.iterrows():
        file_path = os.path.join(ESC50_AUDIO_PATH, row["filename"])
        file_path = file_path.replace("\\", "/")
        category = row["category"]
        
        if category in BIO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

        elif category in ANTRO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

        elif category in GEO_CLASSES:
            dicio.setdefault(category, []).append(file_path)

            
    return dicio

In [ ]:
def generate_mix(proportions, map, bio, antro, geo, output_filename):
    """
    Gera um áudio sintético misturando clipes puros com base nas proporções.
    """
    print(f"\nGerando mix para: {output_filename}")
    print(f"Proporções: {proportions}")
    
    final_signal = np.zeros(DURATION_SAMPLES)
    
    # Adiciona Biofonia
    if proportions['bio'] > 0:
        bio_list = map[bio]
        file_to_load = random.choice(bio_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions['bio']
        print(f"  + Áudio BIO:   {os.path.basename(file_to_load)}")

    # Adiciona Antropofonia
    if proportions['antro'] > 0:
        antro_list = map[antro]
        file_to_load = random.choice(antro_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions['antro']
        print(f"  + Áudio ANTRO: {os.path.basename(file_to_load)}")

    # Adiciona Geofonia
    if proportions['geo'] > 0:
        geo_list = map[geo]
        file_to_load = random.choice(geo_list)
        signal = load_audio(file_to_load, DURATION_SAMPLES)
        signal_norm = normalize_rms(signal)
        final_signal += signal_norm * proportions['geo']
        print(f"  + Áudio GEO:   {os.path.basename(file_to_load)}")
        
    # Normaliza o sinal final para evitar clipping (ruído)
    max_val = np.max(np.abs(final_signal))
    if max_val > 1.0:
        final_signal = final_signal / max_val
    
    # Salva 
    sf.write(output_filename, final_signal, SAMPLE_RATE)

In [8]:
map = build_file_lists()
for key in map.keys():
    if key in BIO_CLASSES:
        print(f"{key} - BIO")
    elif key in ANTRO_CLASSES:
        print(f'{key} - ANTRO')
    else:
        print(f'{key} - GEO')    

dog - BIO
chirping_birds - BIO
vacuum_cleaner - ANTRO
thunderstorm - GEO
door_wood_knock - ANTRO
can_opening - ANTRO
crow - BIO
clapping - BIO
fireworks - ANTRO
chainsaw - ANTRO
airplane - ANTRO
mouse_click - ANTRO
pouring_water - ANTRO
train - ANTRO
sheep - BIO
water_drops - GEO
church_bells - ANTRO
clock_alarm - ANTRO
keyboard_typing - ANTRO
wind - GEO
footsteps - ANTRO
frog - BIO
cow - BIO
brushing_teeth - ANTRO
car_horn - ANTRO
crackling_fire - GEO
helicopter - ANTRO
drinking_sipping - ANTRO
rain - GEO
insects - BIO
laughing - BIO
hen - BIO
engine - ANTRO
breathing - BIO
crying_baby - BIO
hand_saw - ANTRO
coughing - BIO
glass_breaking - ANTRO
snoring - BIO
toilet_flush - ANTRO
pig - BIO
washing_machine - ANTRO
clock_tick - ANTRO
sneezing - BIO
rooster - BIO
sea_waves - GEO
siren - ANTRO
cat - BIO
door_wood_creaks - ANTRO
crickets - BIO


In [ ]:
proporcoes = {
    'bio': 0.0,   
    'antro': 0.2, 
    'geo': 0.8    
}

bio = 'cat'
antro = 'car_horn'
geo = 'rain'

generate_mix(proporcoes, 
            map, bio, antro, geo, 
            output_filename="teste_0B_20A_80G.wav")


Gerando mix para: teste_0B_20A_80G.wav
Proporções: {'bio': 0.0, 'antro': 0.2, 'geo': 0.8}
  + Áudio ANTRO: 1-17124-A-43.wav
  + Áudio GEO:   5-202898-A-10.wav
